In [10]:
# ==============================================================================
# CELL 1: Package Installations
# ==============================================================================
# omnivoice (OmniVoice TTS) requires transformers>=5.3.0. It is installed in
# THIS cell, together with everything else, so every later cell (ASR, LLM,
# TTS) sees the same transformers version from the start.
#
# IMPORTANT: if you previously ran this notebook with an older transformers
# already imported in this kernel, go to Runtime > Restart session before
# re-running from the top. Upgrading a package with pip has no effect on
# code that already imported the old version in the current process.
!pip install -q -U "transformers>=5.3.0" torchaudio librosa accelerate soundfile jiwer
!pip install -q -U peft bitsandbytes torchao
!pip install -q omnivoice


In [11]:
# ==============================================================================
# CELL 2: System Verification & Device Setup
# ==============================================================================
import os
import re
import gc
import time
import base64
import torch
import torchaudio
import librosa
import numpy as np
import pandas as pd
from IPython.display import HTML, Audio, display
from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Active Execution Device: {device}")
if device == "cuda":
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Allocated: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Active Execution Device: cuda
   GPU Model: Tesla T4
   VRAM Allocated: 15.64 GB


In [12]:
# ==============================================================================
# CELL 3: Fidel Text Normalization Engine
# ==============================================================================
def normalize_amharic_text(text: str) -> str:
    """Standardizes Amharic text by folding Ge'ez homophones and removing punctuation."""
    if not text:
        return ""

    text = re.sub(r'\[AMH\]', '', text)

    # Homophone folding
    text = re.sub(r'[ሐኀኻⶠ]', 'ሀ', text)
    text = re.sub(r'[ሑኁኹ]', 'ሁ', text)
    text = re.sub(r'[ሒኂኺ]', 'ሂ', text)
    text = re.sub(r'[ሓኃኻ]', 'ሃ', text)
    text = re.sub(r'[ሔኄኼ]', 'ሄ', text)
    text = re.sub(r'[ሕኅኽ]', 'ህ', text)
    text = re.sub(r'[ሖኆኾ]', 'ሆ', text)

    text = re.sub(r'[ሠ]', 'ሰ', text)
    text = re.sub(r'[ሡ]', 'ሱ', text)
    text = re.sub(r'[ሢ]', 'ሲ', text)
    text = re.sub(r'[ሣ]', 'ሳ', text)
    text = re.sub(r'[ሤ]', 'ሴ', text)
    text = re.sub(r'[ሥ]', 'ስ', text)
    text = re.sub(r'[ሦ]', 'ሶ', text)

    text = re.sub(r'[ዐ]', 'አ', text)
    text = re.sub(r'[ዑ]', 'ኡ', text)
    text = re.sub(r'[ዒ]', 'ኢ', text)
    text = re.sub(r'[ዓ]', 'ኣ', text)
    text = re.sub(r'[ዔ]', 'ኤ', text)
    text = re.sub(r'[ዕ]', 'እ', text)
    text = re.sub(r'[ዖ]', 'ኦ', text)

    text = re.sub(r'[ፀ]', 'ጸ', text)
    text = re.sub(r'[ፁ]', 'ጹ', text)
    text = re.sub(r'[ፂ]', 'ጺ', text)
    text = re.sub(r'[ፃ]', 'ጻ', text)
    text = re.sub(r'[ፄ]', 'ጼ', text)
    text = re.sub(r'[ፅ]', 'ጽ', text)
    text = re.sub(r'[ፆ]', 'ጾ', text)

    text = re.sub(r'[\u1360-\u1368\?\!\.\,\-\"\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Normalization Engine Ready.")

✅ Normalization Engine Ready.


In [13]:
# ==============================================================================
# CELL 4: Browser Microphone Audio Recorder Widget (HTML5/JS with Native WAV)
# ==============================================================================
from google.colab import output

RECORD_HTML = """
<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; width: 360px; text-align: center; font-family: sans-serif;">
    <h4>🎙️ Amharic Voice Assistant</h4>
    <button id="startBtn" style="background-color: #4CAF50; color: white; padding: 10px 20px; border: none; border-radius: 5px; cursor: pointer; margin-right: 10px;">Record</button>
    <button id="stopBtn" style="background-color: #f44336; color: white; padding: 10px 20px; border: none; border-radius: 5px; cursor: pointer;" disabled>Stop</button>
    <p id="status" style="margin-top: 10px; color: #555; font-size: 14px;">Status: Ready</p>
</div>

<script>
let audioContext;
let mediaStream;
let recProcessor;
let audioBuffers = [];

function encodeWAV(samples, sampleRate) {
    let buffer = new ArrayBuffer(44 + samples.length * 2);
    let view = new DataView(buffer);

    /* RIFF identifier */
    for (let i = 0; i < 4; i++) view.setUint8(i, "RIFF".charCodeAt(i));
    /* RIFF chunk length */
    view.setUint32(4, 36 + samples.length * 2, true);
    /* RIFF type */
    for (let i = 0; i < 4; i++) view.setUint8(8 + i, "WAVE".charCodeAt(i));
    /* format chunk identifier */
    for (let i = 0; i < 4; i++) view.setUint8(12 + i, "fmt ".charCodeAt(i));
    /* format chunk length */
    view.setUint32(16, 16, true);
    /* sample format (PCM) */
    view.setUint16(20, 1, true);
    /* channel count (mono) */
    view.setUint16(22, 1, true);
    /* sample rate */
    view.setUint32(24, sampleRate, true);
    /* byte rate */
    view.setUint32(28, sampleRate * 2, true);
    /* block align */
    view.setUint16(32, 2, true);
    /* bits per sample */
    view.setUint16(34, 16, true);
    /* data chunk identifier */
    for (let i = 0; i < 4; i++) view.setUint8(36 + i, "data".charCodeAt(i));
    /* data chunk length */
    view.setUint32(40, samples.length * 2, true);

    /* PCM sample encoding */
    let index = 44;
    for (let i = 0; i < samples.length; i++) {
        let s = Math.max(-1, Math.min(1, samples[i]));
        view.setInt16(index, s < 0 ? s * 0x8000 : s * 0x7FFF, true);
        index += 2;
    }
    return new Blob([view], { type: 'audio/wav' });
}

document.getElementById('startBtn').onclick = async () => {
    try {
        mediaStream = await navigator.mediaDevices.getUserMedia({ audio: true });
        audioContext = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: 16000 });
        let source = audioContext.createMediaStreamSource(mediaStream);

        recProcessor = audioContext.createScriptProcessor(4096, 1, 1);
        audioBuffers = [];

        recProcessor.onaudioprocess = (e) => {
            let inputBuffer = e.inputBuffer.getChannelData(0);
            audioBuffers.push(new Float32Array(inputBuffer));
        };

        source.connect(recProcessor);
        recProcessor.connect(audioContext.destination);

        document.getElementById('startBtn').disabled = true;
        document.getElementById('stopBtn').disabled = false;
        document.getElementById('status').innerText = "🔴 Recording... Speak in Amharic";
    } catch (err) {
        document.getElementById('status').innerText = "❌ Mic Error: " + err.message;
    }
};

document.getElementById('stopBtn').onclick = async () => {
    try {
        recProcessor.disconnect();
        mediaStream.getTracks().forEach(track => track.stop());

        let totalLength = audioBuffers.reduce((acc, buf) => acc + buf.length, 0);
        let pcmData = new Float32Array(totalLength);
        let offset = 0;
        for (let buf of audioBuffers) {
            pcmData.set(buf, offset);
            offset += buf.length;
        }

        let wavBlob = encodeWAV(pcmData, 16000);
        let reader = new FileReader();
        reader.readAsDataURL(wavBlob);
        reader.onloadend = () => {
            let base64Audio = reader.result.split(',')[1];
            document.getElementById('status').innerText = "⏳ Processing in Python...";

            google.colab.kernel.invokeFunction('notebook.process_audio', [base64Audio], {})
                .then(() => {
                    document.getElementById('status').innerText = "✅ Completed! Ready for next input.";
                })
                .catch(err => {
                    document.getElementById('status').innerText = "❌ Python Error (See cell output below)";
                    console.error("Colab Kernel Error:", err);
                });
        };
    } catch (err) {
        document.getElementById('status').innerText = "❌ Encoding Error: " + err.message;
    }

    document.getElementById('startBtn').disabled = false;
    document.getElementById('stopBtn').disabled = true;
};
</script>
"""

def record_microphone_audio():
    """Renders the HTML widget."""
    display(HTML(RECORD_HTML))

In [14]:
# ==============================================================================
# CELL 5: Load ASR Model (badrex/Ethio-ASR-amharic) on CPU to Save VRAM
# ==============================================================================
import gc
import torch
import numpy as np
import librosa
from transformers import AutoProcessor, AutoModelForCTC

# Free up any fragmented CUDA memory before loading
gc.collect()
torch.cuda.empty_cache()

print("⏳ Loading ASR Model (badrex/Ethio-ASR-amharic) on CPU...")
asr_processor = AutoProcessor.from_pretrained("badrex/Ethio-ASR-amharic")

# Force the ASR model to CPU to completely preserve GPU VRAM for the 7B LLM
asr_model = AutoModelForCTC.from_pretrained("badrex/Ethio-ASR-amharic").to("cpu")
asr_model.eval()
print("✅ ASR Model Loaded on CPU.")

def transcribe_audio_array(audio_array: np.ndarray, sampling_rate: int = 16000) -> str:
    """Transcribes raw input audio array to Amharic text utilizing CPU inference."""
    if sampling_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
        sampling_rate = 16000

    # Run preprocessing and inference on CPU
    inputs = asr_processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
    inputs = {k: v.to("cpu") for k, v in inputs.items()}

    with torch.inference_mode():
        logits = asr_model(**inputs).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        raw_transcription = asr_processor.batch_decode(predicted_ids)[0]

    return normalize_amharic_text(raw_transcription)

⏳ Loading ASR Model (badrex/Ethio-ASR-amharic) on CPU...


Loading weights:   0%|          | 0/796 [00:00<?, ?it/s]

✅ ASR Model Loaded on CPU.


In [15]:
# ==============================================================================
# CELL 6: Load LLM Engine (EthioNLP Amharic-LLAMA + LoRA Adapter in 4-bit) - Uncapped
# ==============================================================================
import re
import torch
import gc

# Clear VRAM before starting
gc.collect()
torch.cuda.empty_cache()

# Ensure bitsandbytes and compatible torchao are installed before PEFT imports
try:
    import bitsandbytes
    import torchao
    from packaging import version
    if version.parse(torchao.__version__) < version.parse("0.16.0"):
        raise ImportError
except ImportError:
    print("Installing compatible bitsandbytes and upgrading torchao...")
    !pip install -q -U bitsandbytes torchao

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("⏳ Loading EthioNLP Amharic-LLAMA Base Model & Adapter in 4-bit...")

base_model_name = "EthioNLP/Amharic-llama-base-model"
adapter_name = "EthioNLP/Amharic-LLAMA-all-data"

# 1. Load Tokenizer & Set Pad Token
llm_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

# Configure 4-bit quantization (NF4) with aggressive CPU offloading properties to avoid VRAM peaks
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
) if device == "cuda" else None

# Configure PyTorch to use expandable segments to reduce memory fragmentation
if device == "cuda":
    import os
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Load 7B Base LLaMA Model (use device_map='auto' for offloaded allocation space)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quantization_config,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

# Safe resizing: Disable mean_resizing to prevent PyTorch from analyzing meta tensors
print("🔄 Safely resizing token embeddings...")
base_model.resize_token_embeddings(51009, mean_resizing=False)

# Clean VRAM once more before loading the adapter
gc.collect()
torch.cuda.empty_cache()

# 3. Attach EthioNLP Instruction-Tuned LoRA Weights
llm_model = PeftModel.from_pretrained(
    base_model,
    adapter_name,
    low_cpu_mem_usage=True,
    dtype=torch.float16 if device == "cuda" else torch.float32
)
llm_model.eval()

print("✅ EthioNLP Amharic-LLAMA Loaded Successfully in 4-bit.")

def sanitize_to_ethiopic(text: str) -> str:
    """Strips foreign token hallucinations and keeps Amharic script."""
    ethiopic_only = re.sub(r'[^\u1200-\u137F\s0-9\.\,\?\!]', '', text)
    return re.sub(r'\s+', ' ', ethiopic_only).strip()

def generate_llm_response(prompt_text: str) -> str:
    """Generates a natural, detailed Amharic response using EthioNLP LLaMA."""

    # Removed sentence-length restriction to allow detailed, natural Amharic conversations
    prompt = f"መመሪያ: ለሚከተለው ጥያቄ ዝርዝር እና ትክክለኛ ምላሽ ስጥ።\nጥያቄ: {prompt_text}\nምላሽ:"

    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.inference_mode():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=256,  # Significantly increased token cap for natural detailed speech
            do_sample=True,      # Enabled sampling for natural flow instead of strict greedy decoding
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=llm_tokenizer.pad_token_id
        )

    full_output = llm_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Extract only text generated after the "ምላሽ:" prefix
    if "ምላሽ:" in full_output:
        raw_response = full_output.split("ምላሽ:")[-1].strip()
    else:
        raw_response = full_output[len(prompt):].strip()

    clean_response = sanitize_to_ethiopic(raw_response)

    if not clean_response:
        clean_response = "ሰላም! እንዴት ልረዳህ እችላለሁ?"

    return clean_response

⏳ Loading EthioNLP Amharic-LLAMA Base Model & Adapter in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

🔄 Safely resizing token embeddings...
✅ EthioNLP Amharic-LLAMA Loaded Successfully in 4-bit.


In [16]:
# ==============================================================================
# CELL 7: High-Quality Amharic Text-to-Speech Engine (via OmniVoice)
# ==============================================================================
import gc
import numpy as np
import torch
from omnivoice import OmniVoice, OmniVoiceGenerationConfig

OMNIVOICE_MODEL_ID = "african-low-resource/omnivoice-amharic"

# Free VRAM before loading. The ASR model is on CPU and the LLM already
# occupies GPU memory in 4-bit, so OmniVoice has to fit in what's left.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_tts_device = "cuda:0" if device == "cuda" else "cpu"
_tts_dtype = torch.float16 if _tts_device.startswith("cuda") else torch.float32

def _load_omnivoice(target_device, target_dtype):
    print(f"⏳ Loading OmniVoice ({OMNIVOICE_MODEL_ID}) on {target_device}...")
    return OmniVoice.from_pretrained(
        OMNIVOICE_MODEL_ID,
        device_map=target_device,
        dtype=target_dtype,
    )

try:
    omnivoice_model = _load_omnivoice(_tts_device, _tts_dtype)
except RuntimeError as e:
    if "out of memory" not in str(e).lower():
        raise
    print("⚠️ Not enough VRAM to share the GPU with the LLM — retrying OmniVoice on CPU.")
    _tts_device, _tts_dtype = "cpu", torch.float32
    gc.collect()
    torch.cuda.empty_cache()
    omnivoice_model = _load_omnivoice(_tts_device, _tts_dtype)

omnivoice_model.eval()
TTS_SAMPLE_RATE = omnivoice_model.sampling_rate or 24000
print(f"✅ OmniVoice Loaded on {_tts_device} (sampling rate: {TTS_SAMPLE_RATE} Hz).")

# Auto-voice mode (no reference audio, no instruct) picks a voice on its own.
# A fixed seed keeps that choice consistent turn-to-turn in this session
# instead of re-rolling a new voice on every response. Drop the
# torch.manual_seed(...) call below if you'd rather have it vary each time.
_TTS_SEED = 42

def synthesize_speech(text: str):
    """Converts Amharic text into speech using OmniVoice (auto voice mode)."""
    if not text or not text.strip():
        print("⚠️ Empty text provided to TTS.")
        return np.zeros(TTS_SAMPLE_RATE), TTS_SAMPLE_RATE

    print(f"🔤 TTS Input (Raw): {text}")

    try:
        torch.manual_seed(_TTS_SEED)
        with torch.inference_mode():
            audio_list = omnivoice_model.generate(
                text=text,
                language="Amharic",
                generation_config=OmniVoiceGenerationConfig(
                    num_step=32,
                    guidance_scale=2.0,
                ),
            )
        audio_data = audio_list[0]
        if isinstance(audio_data, torch.Tensor):
            audio_data = audio_data.detach().cpu().float().numpy()
        audio_data = np.asarray(audio_data).squeeze()
        return audio_data, TTS_SAMPLE_RATE
    except Exception as e:
        print(f"⚠️ TTS generation error ({e}).")
        return np.zeros(TTS_SAMPLE_RATE), TTS_SAMPLE_RATE


⏳ Loading OmniVoice (african-low-resource/omnivoice-amharic) on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✅ OmniVoice Loaded on cuda:0 (sampling rate: 24000 Hz).


In [17]:
# ==============================================================================
# CELL 8: Audio Pipeline Orchestrator & Colab Callback Registration
# ==============================================================================
from google.colab import output
import io
import soundfile as sf
import base64

def process_audio(base64_audio_str: str):
    """Backend processor callback triggered when the user stops recording."""
    try:
        print("\n--- New Voice Interaction ---")

        # 1. Decode base64 audio payload
        audio_bytes = base64.b64decode(base64_audio_str)

        # 2. Read the WAV data
        data, samplerate = sf.read(io.BytesIO(audio_bytes))

        # 3. Transcribe speech using badrex/Ethio-ASR-amharic
        print("🎤 Transcribing audio...")
        user_query = transcribe_audio_array(data, sampling_rate=samplerate)
        print(f"🗣️ User (Transcribed): {user_query}")

        if not user_query.strip():
            user_query = "ሰላም"
            print("⚠️ Transcription was empty, defaulting query to 'ሰላም'")

        # 4. Generate response using EthioNLP Amharic-LLAMA
        print("🧠 Generating AI response...")
        ai_response = generate_llm_response(user_query)
        print(f"🤖 Assistant: {ai_response}")

        # 5. Synthesize TTS speech using african-low-resource/omnivoice-amharic
        print("🔊 Generating audio speech output...")
        speech_wave, tts_sr = synthesize_speech(ai_response)

        # 6. Auto-play the response in the notebook output
        display(Audio(speech_wave, rate=tts_sr, autoplay=True))

    except Exception as e:
        print(f"❌ Error in audio processing pipeline: {e}")

# Register the Python function as a callback accessible by JavaScript
output.register_callback('notebook.process_audio', process_audio)
print("✅ Orchestration Pipeline & Colab Callback Registered.")

✅ Orchestration Pipeline & Colab Callback Registered.


In [19]:
# ==============================================================================
# CELL 9: Run Voice Assistant Session
# ==============================================================================
# Render the recorder widget. Clicking 'Stop' automatically triggers Python processing.
record_microphone_audio()


--- New Voice Interaction ---
🎤 Transcribing audio...
🗣️ User (Transcribed): ሰላም ጤና አይስጥልን እንዴት እና
🧠 Generating AI response...
🤖 Assistant: ሰላም እና ጤና ይስጥልኝ፣ እኔ የ ቋንቋ ሞዴል ነኝ እና በባህላዊ ስሜት ጤናን ከመለኮት ወይም ከማንኛውም ሌላ ኃይል መቀበል አልችልም. ጤናን ወደ ህይወትዎ መጨመር የምችለው አንድ መንገድ በአመጋገብዎ፣ በአካል ብቃት እንቅስቃሴዎ እና በአጠቃላይ ጤናማ የአኗኗር ዘይቤ መኖር ነው።
🔊 Generating audio speech output...
🔤 TTS Input (Raw): ሰላም እና ጤና ይስጥልኝ፣ እኔ የ ቋንቋ ሞዴል ነኝ እና በባህላዊ ስሜት ጤናን ከመለኮት ወይም ከማንኛውም ሌላ ኃይል መቀበል አልችልም. ጤናን ወደ ህይወትዎ መጨመር የምችለው አንድ መንገድ በአመጋገብዎ፣ በአካል ብቃት እንቅስቃሴዎ እና በአጠቃላይ ጤናማ የአኗኗር ዘይቤ መኖር ነው።



--- New Voice Interaction ---
🎤 Transcribing audio...
🗣️ User (Transcribed): ዛሬ ቀኑ እንደ ኢትዮጵያ አቆጣጠር መስከረም 172019 ነው ከዚህ በመነሳት የአየር ሁኔታው ምን ይመስላል
🧠 Generating AI response...
🤖 Assistant: አየሩ ሞቃት ይሆናል። የሙቀት መጠኑ 27 ይሆናል.
🔊 Generating audio speech output...
🔤 TTS Input (Raw): አየሩ ሞቃት ይሆናል። የሙቀት መጠኑ 27 ይሆናል.



--- New Voice Interaction ---
🎤 Transcribing audio...
🗣️ User (Transcribed): መስከረም 17 19 ኣ ምት የምን በኣል ይከበራል በኢትዮጵያ
🧠 Generating AI response...
🤖 Assistant: በርግጥ እ.ኤ.አ. በሴፕቴምበር 17 ቀን 2019 ዓለም አቀፍ የህፃናት ካንሰር ግንዛቤን ለማሳደግ በዓለም ዙሪያ ያሉ ሰዎች ቢጫ ይለብሳሉ
🔊 Generating audio speech output...
🔤 TTS Input (Raw): በርግጥ እ.ኤ.አ. በሴፕቴምበር 17 ቀን 2019 ዓለም አቀፍ የህፃናት ካንሰር ግንዛቤን ለማሳደግ በዓለም ዙሪያ ያሉ ሰዎች ቢጫ ይለብሳሉ



--- New Voice Interaction ---
🎤 Transcribing audio...
🗣️ User (Transcribed): ሴፕቴምበር 17 ሳይሆን መስከረም 172019 አ ምህት እንደ ኢትዮጵያ አቆጣጠር የምን ክብረ በኣል ይከበራል
🧠 Generating AI response...
🤖 Assistant: በእርግጥ የኢትዮጵያ የ19ኛው ክፍለ ዘመን መጀመሪያ ዋና ከተማ በሆነችው በአጼ ምኒልክ ዘመነ መንግሥት የፀደቀው ይህ ቀን በጎርጎርያን ዘዴ መሠረት ቅዳሜ ጥቅምት 23፣ 1899 ጎርጎሪያን ነው። በኢትዮጵያ ግን በወርቅ ሚዛን አቆጣጠር መሠረት ቅዳሜ መስከረም 7፣ 1900 ዓም ነበር።
🔊 Generating audio speech output...
🔤 TTS Input (Raw): በእርግጥ የኢትዮጵያ የ19ኛው ክፍለ ዘመን መጀመሪያ ዋና ከተማ በሆነችው በአጼ ምኒልክ ዘመነ መንግሥት የፀደቀው ይህ ቀን በጎርጎርያን ዘዴ መሠረት ቅዳሜ ጥቅምት 23፣ 1899 ጎርጎሪያን ነው። በኢትዮጵያ ግን በወርቅ ሚዛን አቆጣጠር መሠረት ቅዳሜ መስከረም 7፣ 1900 ዓም ነበር።



--- New Voice Interaction ---
🎤 Transcribing audio...
🗣️ User (Transcribed): የመስቀል በኣል መቼ ይከበራል ቀኑ ስንት ነው
🧠 Generating AI response...
🤖 Assistant: መስቀል በዓል በቢሾፍቱ ከተማ ጥቅምት አስራ አንዱ ይከበራል። በዓሉ ለአስራ አንድ ቀናት የሚቆይ ሲሆን በአዲስ አበባ ላይ ያለው ድባብ በበዓሉ ቀን ጥቅምት አንድ ይሆናል።
🔊 Generating audio speech output...
🔤 TTS Input (Raw): መስቀል በዓል በቢሾፍቱ ከተማ ጥቅምት አስራ አንዱ ይከበራል። በዓሉ ለአስራ አንድ ቀናት የሚቆይ ሲሆን በአዲስ አበባ ላይ ያለው ድባብ በበዓሉ ቀን ጥቅምት አንድ ይሆናል።
